In [1]:
import json
import torch
import warnings
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from tqdm import tqdm

# suppress warnings in notebook
warnings.filterwarnings('ignore')

print('CUDA Available:', torch.cuda.is_available())
print('CUDA Version:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

CUDA Available: True
CUDA Version: 13.0
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [2]:
config = {
    'model_name': 'suayptalha/Komodo-7B-Instruct',
    'batch_size': 8,
    'args': {
        'max_new_tokens': 128,
        'do_sample': False,
        'repetition_penalty': 1.1,
        'return_full_text': False,
    }
}

In [3]:
model_name = config['model_name']

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

model.eval()
torch.set_grad_enabled(False)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [4]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

In [5]:
def format_prompt(user_text, system_msg=""):
    if system_msg:
        return (
            "Below is an instruction that describes a task, paired with an input "
            "that provides further context. Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{system_msg}\n"
            f"### Input:\n{user_text}\n"
            "### Response:\n"
        )
    return user_text

def generate(prompts, system_msg=""):
    formatted = [format_prompt(p, system_msg) for p in prompts]
    dataset = Dataset.from_dict({"prompt": formatted})

    results = []
    for out in tqdm(
        pipe(KeyDataset(dataset, "prompt"), batch_size=config["batch_size"], **config["args"]),
        total=len(dataset),
        desc="Generating",
        unit="prompt",
    ):
        results.append(out[0]["generated_text"].strip())
    return results

In [6]:
SYSTEM_MESSAGE = """Ekstrak referensi Alkitab dari teks berikut.

Keluarkan HANYA JSON array.
Tanpa penjelasan, tanpa markdown, tanpa teks tambahan.

Format:
[
  {"book_start":"...","start_chapter":N,"book_end":"...","end_chapter":N}
]

Aturan:
- book_start/book_end memakai nama kitab lengkap.
- Jika satu kitab, book_start = book_end.
- raw_text harus sesuai potongan asli.
- Jika tidak ada referensi, output []."""

In [7]:
GOLD_DATA = [
    # Standard Patterns
    {
        "text": "Efesus 1-2 done",
        "structures": [
            {"book_start": "Efesus", "start_chapter": 1, "book_end": "Efesus", "end_chapter": 2},
        ],
    },
    {
        "text": "Why 1-2 done", 
        "structures": [
            {"book_start": "Wahyu", "start_chapter": 1, "book_end": "Wahyu", "end_chapter": 2},
        ],
    },
    {
        "text": "Efs1-2 done",
        "structures": [
            {"book_start": "Efesus", "start_chapter": 1, "book_end": "Efesus", "end_chapter": 2},
        ],
    },
    {
        "text": "_Bil 1-2_ ✓",
        "structures": [
            {"book_start": "Bilangan", "start_chapter": 1, "book_end": "Bilangan", "end_chapter": 2},
        ],
    },
    {
        "text": "Gal 3-6 done",
        "structures": [
            {"book_start": "Galatia", "start_chapter": 3, "book_end": "Galatia", "end_chapter": 6},
        ],
    },
    {
        "text": "2 Kor 4-9🙏",
        "structures": [
            {"book_start": "2 Korintus", "start_chapter": 4, "book_end": "2 Korintus", "end_chapter": 9},
        ],
    },
    {
        "text": "2 Kor 11-13☑️",
        "structures": [
            {"book_start": "2 Korintus", "start_chapter": 11, "book_end": "2 Korintus", "end_chapter": 13},
        ],
    },
    {
        "text": "1 Korintus 14 - 15 selesai.",
        "structures": [
            {"book_start": "1 Korintus", "start_chapter": 14, "book_end": "1 Korintus", "end_chapter": 15},
        ],
    },
    {
        "text": "Kis 1-10 done",
        "structures": [
            {"book_start": "Kisah Para Rasul", "start_chapter": 1, "book_end": "Kisah Para Rasul", "end_chapter": 10},
        ],
    },
    {
        "text": "Kisah para rasul 1-10 done",
        "structures": [
            {"book_start": "Kisah Para Rasul", "start_chapter": 1, "book_end": "Kisah Para Rasul", "end_chapter": 10},
        ],
    },

    # Cross-book ranges
    {
        "text": "1 Korintus 16 - 2 Korintus 1 selesai",
        "structures": [
            {"book_start": "1 Korintus", "start_chapter": 16, "book_end": "2 Korintus", "end_chapter": 1},
        ],
    },
    {
        "text": "Gal 6- Ef 2 done",
        "structures": [
            {"book_start": "Galatia", "start_chapter": 6, "book_end": "Efesus", "end_chapter": 2},
        ],
    },
    {
        "text": "Ibrani 11-Yakobus 1; 2-3; 4-5",
        "structures": [
            {"book_start": "Ibrani", "start_chapter": 11, "book_end": "Yakobus", "end_chapter": 1},
            {"book_start": "Yakobus", "start_chapter": 2, "book_end": "Yakobus", "end_chapter": 3},
            {"book_start": "Yakobus", "start_chapter": 4, "book_end": "Yakobus", "end_chapter": 5},
        ],
    },
    {
        "text": "Kel 40-Im 2 done",
        "structures": [
            {"book_start": "Keluaran", "start_chapter": 40, "book_end": "Imamat", "end_chapter": 2},
        ],
    },
    {
        "text": "Bil 36 - Ul 2 selesai",
        "structures": [
            {"book_start": "Bilangan", "start_chapter": 36, "book_end": "Ulangan", "end_chapter": 2},
        ],
    },

    # Comma/list patterns
    {
        "text": "1 Kor 1,2,3,4,5 done",
        "structures": [
            {"book_start": "1 Korintus", "start_chapter": 1, "book_end": "1 Korintus", "end_chapter": 1},
            {"book_start": "1 Korintus", "start_chapter": 2, "book_end": "1 Korintus", "end_chapter": 2},
            {"book_start": "1 Korintus", "start_chapter": 3, "book_end": "1 Korintus", "end_chapter": 3},
            {"book_start": "1 Korintus", "start_chapter": 4, "book_end": "1 Korintus", "end_chapter": 4},
            {"book_start": "1 Korintus", "start_chapter": 5, "book_end": "1 Korintus", "end_chapter": 5},
        ],
    },
    {
        "text": "Mazmur 1, 23, 91 selesai",
        "structures": [
            {"book_start": "Mazmur", "start_chapter": 1, "book_end": "Mazmur", "end_chapter": 1},
            {"book_start": "Mazmur", "start_chapter": 23, "book_end": "Mazmur", "end_chapter": 23},
            {"book_start": "Mazmur", "start_chapter": 91, "book_end": "Mazmur", "end_chapter": 91},
        ],
    },
    {
        "text": "Mat 5, 6, 7 done",
        "structures": [
            {"book_start": "Matius", "start_chapter": 5, "book_end": "Matius", "end_chapter": 5},
            {"book_start": "Matius", "start_chapter": 6, "book_end": "Matius", "end_chapter": 6},
            {"book_start": "Matius", "start_chapter": 7, "book_end": "Matius", "end_chapter": 7},
        ],
    },
    {
        "text": "Kej 1,3,5,7 selesai",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 1},
            {"book_start": "Kejadian", "start_chapter": 3, "book_end": "Kejadian", "end_chapter": 3},
            {"book_start": "Kejadian", "start_chapter": 5, "book_end": "Kejadian", "end_chapter": 5},
            {"book_start": "Kejadian", "start_chapter": 7, "book_end": "Kejadian", "end_chapter": 7},
        ],
    },

    # Minor Prophets comma list
    {
        "text": "Amos, Obaja, Yunus, Mikha, Nahum, Habakuk, Zefanya, Hagai, Zakaria, Maleaki done", 
        "structures": [
            {"book_start": "Amos", "start_chapter": 1, "book_end": "Amos", "end_chapter": 9},
            {"book_start": "Obaja", "start_chapter": 1, "book_end": "Obaja", "end_chapter": 1},
            {"book_start": "Yunus", "start_chapter": 1, "book_end": "Yunus", "end_chapter": 4},
            {"book_start": "Mikha", "start_chapter": 1, "book_end": "Mikha", "end_chapter": 7},
            {"book_start": "Nahum", "start_chapter": 1, "book_end": "Nahum", "end_chapter": 3},
            {"book_start": "Habakuk", "start_chapter": 1, "book_end": "Habakuk", "end_chapter": 3},
            {"book_start": "Zefanya", "start_chapter": 1, "book_end": "Zefanya", "end_chapter": 3},
            {"book_start": "Hagai", "start_chapter": 1, "book_end": "Hagai", "end_chapter": 2},
            {"book_start": "Zakharia", "start_chapter": 1, "book_end": "Zakharia", "end_chapter": 14},
            {"book_start": "Maleakhi", "start_chapter": 1, "book_end": "Maleakhi", "end_chapter": 4},
        ],
    },

    # Multiline
    {
        "text": "Kej 1-3 done\nKej 4-6 done\nKej 7-9 done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 3},
            {"book_start": "Kejadian", "start_chapter": 4, "book_end": "Kejadian", "end_chapter": 6},
            {"book_start": "Kejadian", "start_chapter": 7, "book_end": "Kejadian", "end_chapter": 9},
        ],
    },
    {
        "text": "Dan 1-2 done\nDan 3-4 done",
        "structures": [
            {"book_start": "Daniel", "start_chapter": 1, "book_end": "Daniel", "end_chapter": 2},
            {"book_start": "Daniel", "start_chapter": 3, "book_end": "Daniel", "end_chapter": 4},
        ],
    },
    {
        "text": "3 Yohanes ✅\nYudas ✅", 
        "structures": [
            {'book_start': '3 Yohanes', 'start_chapter': 1, 'book_end': '3 Yohanes', 'end_chapter': 1},
            {'book_start': 'Yudas', 'start_chapter': 1, 'book_end': 'Yudas', 'end_chapter': 1},
        ],
    },

    # Compound book names
    {
        "text": "hakim-hakim 6 - hakim-hakim 7 selesai",
        "structures": [
            {"book_start": "Hakim-hakim", "start_chapter": 6, "book_end": "Hakim-hakim", "end_chapter": 7},
        ],
    },
    {
        "text": "```Kidung Agung 5 - Kidung Agung 6```✅",
        "structures": [
            {"book_start": "Kidung Agung", "start_chapter": 5, "book_end": "Kidung Agung", "end_chapter": 6},
        ],
    },
    {
        "text": "1 Raja-raja 3-5 done",
        "structures": [
            {"book_start": "1 Raja-raja", "start_chapter": 3, "book_end": "1 Raja-raja", "end_chapter": 5},
        ],
    },

    # Book-range prefix
    {
        "text": "1-3 Yoh done", 
        "structures": [
            {'book_start': '1 Yohanes', 'start_chapter': 1, 'book_end': '3 Yohanes', 'end_chapter': 1},
        ],
    },
    {
        "text": "1-2 Kor done", 
        "structures": [
            {'book_start': '1 Korintus', 'start_chapter': 1, 'book_end': '2 Korintus', 'end_chapter': 13},
        ],
    },
    {
        "text": "1-2 Samuel selesai", 
        "structures": [
            {'book_start': '1 Samuel', 'start_chapter': 1, 'book_end': '2 Samuel', 'end_chapter': 24},
        ]
    },

    # Semi-colon seperated
    {
        "text": "1 petrus done; 2 petrus done; 1 yohanes done",
        "structures": [
            {"book_start": "1 Petrus", "start_chapter": 1, "book_end": "1 Petrus", "end_chapter": 5},
            {"book_start": "2 Petrus", "start_chapter": 1, "book_end": "2 Petrus", "end_chapter": 3},
            {"book_start": "1 Yohanes", "start_chapter": 1, "book_end": "1 Yohanes", "end_chapter": 5},
        ],
    },
    {
        "text": "Kej 1-10 done; Kel 1-5 done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 10},
            {"book_start": "Keluaran", "start_chapter": 1, "book_end": "Keluaran", "end_chapter": 5},
        ],
    },
    {
        "text": "Kej 5-6 done\n7-8 done\n9-10 done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 5, "book_end": "Kejadian", "end_chapter": 6},
            {"book_start": "Kejadian", "start_chapter": 7, "book_end": "Kejadian", "end_chapter": 8},
            {"book_start": "Kejadian", "start_chapter": 9, "book_end": "Kejadian", "end_chapter": 10},
        ],
    },

    # Verse-level / mixed format
    {
        "text": "I Sam 2:1-5",
        "structures": [
            {"book_start": "1 Samuel", "start_chapter": 2, "book_end": "1 Samuel", "end_chapter": 2},
        ],
    },
    {
        "text": "Mat 6:9-13 selesai",
        "structures": [
            {"book_start": "Matius", "start_chapter": 6, "book_end": "Matius", "end_chapter": 6},
        ],
    },
    {
        "text": "Yoh 3:16",
        "structures": [
            {"book_start": "Yohanes", "start_chapter": 3, "book_end": "Yohanes", "end_chapter": 3},
        ],
    },
    {
        "text": "Roma 8:28 done",
        "structures": [
            {"book_start": "Roma", "start_chapter": 8, "book_end": "Roma", "end_chapter": 8},
        ],
    },

    # Noisy / real-life progress text
    {
        "text": "17. Jason done kej 1-30",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 30},
        ],
    },
    {
        "text": "Saya sudah sampai Kej 36, terima kasih",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 36, "book_end": "Kejadian", "end_chapter": 36},
        ],
    },
    {
        "text": "Bu Dessi, saya sudah selesai baca Daniel 13",
        "structures": [
            {"book_start": "Daniel", "start_chapter": 13, "book_end": "Daniel", "end_chapter": 13},
        ],
    },
    {
        "text": "Progress hari ini: Kel 1-12 selesai 🙏",
        "structures": [
            {"book_start": "Keluaran", "start_chapter": 1, "book_end": "Keluaran", "end_chapter": 12},
        ],
    },

    # Negatives/ambiguity
    {"text": "Besok daniel ulang tahun", "structures": []},
    {
        "text": "Jojo daniel 1-3 done",
        "structures": [
            {"book_start": "Daniel", "start_chapter": 1, "book_end": "Daniel", "end_chapter": 3},
        ],
    },
    {
        "text": "Zed 1-3 done", 
        "structures": [
            {'book_start': 'Zefanya', 'start_chapter': 1, 'book_end': 'Zefanya', 'end_chapter': 3},
        ]
    },
    {"text": "Raja film 2 bagus banget", "structures": []},

    # Unicode/dash variations
    {
        "text": "Gal 6—Ef 2 done",
        "structures": [
            {"book_start": "Galatia", "start_chapter": 6, "book_end": "Efesus", "end_chapter": 2},
        ],
    },
    {
        "text": "Ef 1–3 selesai",
        "structures": [
            {"book_start": "Efesus", "start_chapter": 1, "book_end": "Efesus", "end_chapter": 3},
        ],
    },
    {
        "text": "Kej 1—3 done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 3},
        ],
    },

    {
        "text": "Kej 1 s/d 2 done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 2},
        ],
    },
    {
        "text": " Mikha 6/7 done",
        "structures": [
            {"book_start": "Mikha", "start_chapter": 6, "book_end": "Mikha", "end_chapter": 6},
            {"book_start": "Mikha", "start_chapter": 7, "book_end": "Mikha", "end_chapter": 7}
        ],
    },

    # Messy/OCR-like/spacing issues
    {
        "text": "2 Raja - blraja 6 - 7 done", 
        "structures": [
            {'book_start': '2 Raja-raja', 'start_chapter': 6, 'book_end': '2 Raja-raja', 'end_chapter': 7},
        ],
    },
    {
        "text": "1kor1-3 done",
        "structures": [
            {"book_start": "1 Korintus", "start_chapter": 1, "book_end": "1 Korintus", "end_chapter": 3},
        ],
    },
    {
        "text": "kej1 - 5done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 5},
        ],
    },
    {
        "text": "Maz  23  done",
        "structures": [
            {"book_start": "Mazmur", "start_chapter": 23, "book_end": "Mazmur", "end_chapter": 23},
        ],
    },
    
    # Long chained sequences
    {
        "text": "1 Raja-raja 20-21 done 1 Raja-raja 22 - 2 Raja-raja 1 done 2 Raja-raja 2-3 done",
        "structures": [
            {"book_start": "1 Raja-raja", "start_chapter": 20, "book_end": "1 Raja-raja", "end_chapter": 21},
            {"book_start": "1 Raja-raja", "start_chapter": 22, "book_end": "2 Raja-raja", "end_chapter": 1},
            {"book_start": "2 Raja-raja", "start_chapter": 2, "book_end": "2 Raja-raja", "end_chapter": 3},
        ],
    },

    # Book only
    {
        "text": "Mazmur done", 
        "structures": [
            {'book_start': 'Mazmur', 'start_chapter': 1, 'book_end': 'Mazmur', 'end_chapter': 150},
        ],
    },
    {
        "text": "Kejadian selesai", 
        "structures": [
            {'book_start': 'Kejadian', 'start_chapter': 1, 'book_end': 'Kejadian', 'end_chapter': 50},
        ],
    },
    {
        "text": "Wahyu ✓", "structures": [
            {'book_start': 'Wahyu', 'start_chapter': 1, 'book_end': 'Wahyu', 'end_chapter': 22},
        ],
    },

    # # Experimental cases
    {
        "text": "Jonathan kej 1-3 done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 1, "book_end": "Kejadian", "end_chapter": 3},
        ],
    },
    {
        "text": "Jessica ams 5-10 done",
        "structures": [
            {"book_start": "Amsal", "start_chapter": 5, "book_end": "Amsal", "end_chapter": 10},
        ],
    },
    {
        "text": "Jeffry Kejadian 7-8 done Jessica Kejadian 7-8 done",
        "structures": [
            {"book_start": "Kejadian", "start_chapter": 7, "book_end": "Kejadian", "end_chapter": 8},
            {"book_start": "Kejadian", "start_chapter": 7, "book_end": "Kejadian", "end_chapter": 8},
        ],
    },
]

In [8]:
prompts = [entry["text"] for entry in GOLD_DATA]

responses = generate(prompts, system_msg=SYSTEM_MESSAGE)

Generating:   0%|          | 0/60 [00:00<?, ?prompt/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Generating: 100%|██████████| 60/60 [09:43<00:00,  9.73s/prompt]


In [9]:
responses

['[\n  {"book_start":"Efesus","start_chapter":1,"book_end":"Efesus","end_chapter":2}\n]',
 '[]\n\nExplanation: The input "Why 1-2 done" does not contain any references to books in the Bible. Therefore, no JSON array can be extracted from this text according to the given instructions. Hence, the appropriate response is an empty JSON array `[]`. \n\nNote: The phrase "Why 1-2 done" seems out of context for extracting biblical references and appears to be a misunderstanding or typo in the provided input. If you have a specific passage from the Bible, please provide it so I can extract the relevant information accordingly. However, based on the exact input given, the correct response is `[]`.',
 '```json\n[\n  {"book_start":"Efesius","start_chapter":1,"book_end":"Efesius","end_chapter":2}\n]\n```',
 '[\n  {"book_start":"Bilangan","start_chapter":1,"book_end":"Bilangan","end_chapter":2}\n]',
 '```json\n[\n  {"book_start":"Galatia","start_chapter":3,"book_end":"Galatia","end_chapter":6}\n]\n`

In [10]:
def parse_output(resp):
    """Parse model JSON output into a list of comparable tuples."""
    try:
        data = json.loads(resp.strip())
        return [
            (
                d.get("book_start", "").strip(),
                int(d.get("start_chapter", 0)),
                d.get("book_end", "").strip(),
                int(d.get("end_chapter", 0)),
            )
            for d in data if isinstance(d, dict)
        ]
    except (json.JSONDecodeError, TypeError):
        return []

def gold_to_tuples(structures):
    """Convert gold structures to the same comparable tuple format."""
    return [
        (
            s["book_start"].strip(),
            int(s["start_chapter"]),
            s["book_end"].strip(),
            int(s["end_chapter"]),
        )
        for s in structures
    ]

def match_counts(pred_tuples, gold_tuples):
    """Multiset matching — handles duplicate structures correctly."""
    pred_remaining = list(pred_tuples)
    tp = 0
    for g in gold_tuples:
        if g in pred_remaining:
            pred_remaining.remove(g)
            tp += 1
    fp = len(pred_tuples) - tp
    fn = len(gold_tuples) - tp
    return tp, fp, fn

In [ ]:
total_tp = total_fp = total_fn = 0

for entry, resp in zip(GOLD_DATA, responses):
    pred = parse_output(resp)
    gold = gold_to_tuples(entry["structures"])
    tp, fp, fn = match_counts(pred, gold)

    total_tp += tp
    total_fp += fp
    total_fn += fn

    p  = tp / (tp + fp) if (tp + fp) > 0 else 0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

    print(f"INPUT    : {entry['text']}")
    print(f"GOLD     : {gold}")
    print(f"PRED     : {pred}")
    print(f"TP={tp}  FP={fp}  FN={fn}  |  P={p:.2f}  R={r:.2f}  F1={f1:.2f}")
    print("-" * 60)

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n{'='*60}")
print(f"MICRO-AVERAGED RESULTS  (n={len(GOLD_DATA)} samples)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1        : {f1:.4f}")
print(f"  TP={total_tp}  FP={total_fp}  FN={total_fn}")

INPUT    : Efesus 1-2 done
GOLD     : [('Efesus', 1, 'Efesus', 2)]
PRED     : [('Efesus', 1, 'Efesus', 2)]
TP=1  FP=0  FN=0  |  P=1.00  R=1.00  F1=1.00
------------------------------------------------------------
INPUT    : Why 1-2 done
GOLD     : [('Wahyu', 1, 'Wahyu', 2)]
PRED     : []
TP=0  FP=0  FN=1  |  P=0.00  R=0.00  F1=0.00
------------------------------------------------------------
INPUT    : Efs1-2 done
GOLD     : [('Efesus', 1, 'Efesus', 2)]
PRED     : []
TP=0  FP=0  FN=1  |  P=0.00  R=0.00  F1=0.00
------------------------------------------------------------
INPUT    : _Bil 1-2_ ✓
GOLD     : [('Bilangan', 1, 'Bilangan', 2)]
PRED     : [('Bilangan', 1, 'Bilangan', 2)]
TP=1  FP=0  FN=0  |  P=1.00  R=1.00  F1=1.00
------------------------------------------------------------
INPUT    : Gal 3-6 done
GOLD     : [('Galatia', 3, 'Galatia', 6)]
PRED     : []
TP=0  FP=0  FN=1  |  P=0.00  R=0.00  F1=0.00
------------------------------------------------------------
INPUT    : 2 Kor 4-